# Establish connection to pawsey0411

In [8]:
import sys
from pathlib import Path

repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from utils.config_utils import check_runtime_readiness, load_pipeline_config, resolve_secrets

Startup version validation. Catch dependency issues fail before scanning data

In [2]:
check_runtime_readiness()

{'fsspec': '2026.3.0',
 's3fs': '2026.3.0',
 'kerchunk': '0.2.10',
 'virtualizarr': '2.6.0'}

## Load pipeline config

Validates the generic kerchunk pipeline schema from config.yaml.

In [9]:
kp = load_pipeline_config("../configs/config.yaml")
print("Schema validation PASSED")
print("Configured source flows:", [f.get("id") for f in kp["source_flows"]])

Schema validation PASSED
Configured source flows: ['ecmwf_daily_nc', 'dpird_final_singleton']


## Get keys

Keys managed on Databricks CLI. [Docs for secret management](https://docs.databricks.com/aws/en/security/secrets/)

In [10]:
ACCESS_KEY, SECRET_KEY = resolve_secrets(kp)
type(ACCESS_KEY)

str

Print top-level directories and objects in weather bucket

In [11]:
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

s3_cfg = kp["s3"]
bucket = s3_cfg["bucket"]

print("S3 config summary:", {
    "endpoint_url": s3_cfg["endpoint_url"],
    "region_name": s3_cfg.get("region_name", "us-east-1"),
    "bucket": bucket,
    "flow_count": len(kp["source_flows"]),
})

s3 = boto3.client(
    "s3",
    endpoint_url=s3_cfg["endpoint_url"],
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name=s3_cfg.get("region_name"),
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)

try:
    print("Buckets visible:", [b["Name"] for b in s3.list_buckets().get("Buckets", [])])
except ClientError as exc:
    print("list_buckets not allowed:", exc.response.get("Error", {}))

resp = s3.list_objects_v2(Bucket=bucket, Delimiter="/")
prefixes = [p["Prefix"] for p in resp.get("CommonPrefixes", [])]
print("Top-level pseudo-directories:", prefixes)

# Top-level objects sample
root_objects = [o["Key"] for o in resp.get("Contents", [])]
print("Top-level objects sample:", root_objects[:20])

S3 config summary: {'endpoint_url': 'https://projects.pawsey.org.au', 'region_name': 'us-east-1', 'bucket': 'weather', 'flow_count': 2}
Buckets visible: ['2025-04-chiles01', '2025-04-chiles01-lsm', '2025-09-chilesaws', 'chiles', 'chiles01', 'chiles01-clean', 'chiles02-0.25', 'chilesaws', 'rtobar-ngas-files', 'weather']
Top-level pseudo-directories: ['DPIRD_clean/', 'FINAL_DPIRD/', 'clean_DPIRD/', 'ecmwf_op_clean/', 'ecmwf_op_minres/', 'ecmwf_op_wswa/', 'gsmap/', 'hmwrarp/', 'hmwrclp/']
Top-level objects sample: ['dataset_DPIRD_utc0.tar.gz', 'dem-9s.tif', 'ecmwf_era5.tar.gz', 'radar.tar.gz']
